# Sales Intelligence — Comprehensive Data Science & Retail Analytics
## Data Science using Python — Applied Superstore Analysis
**Author:** Data Science & Analytics Team  
**Dataset:** Sample - Superstore (Retail Transaction Records)  
**Environment:** Python 3.11+ / Pandas / NumPy / Matplotlib / Seaborn / SciPy / Scikit-Learn

---

## 1. Problem Statement & Business Context
A national retail superstore chain generates millions of dollars in revenue across three primary categories: **Furniture**, **Office Supplies**, and **Technology**.
However, aggressive discounting and regional variances have resulted in significant profit leakage.

### Objectives:
1. Audit data quality (missing values, schema consistency, duplicates).
2. Preprocess data and engineer domain-specific features (Order Processing Days, Time Dimensions, Profit Margins).
3. Conduct thorough Exploratory Data Analysis (EDA) on sales, profit, regions, and products.
4. Perform statistical hypothesis testing and feature correlation.
5. Build a time-series forecasting regression model with strictly chronological train/test separation.
6. Formulate actionable, data-driven business recommendations.

## 2. Library Imports & Environment Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10
print("Libraries successfully imported.")

## 3. Dataset Loading & Inspection

In [ ]:
# Resolve dataset path safely
csv_path = '../backend/data/Sample - Superstore.csv'
if not os.path.exists(csv_path):
    csv_path = 'Sample - Superstore.csv'

# Load with encoding fallback
try:
    df_raw = pd.read_csv(csv_path, encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv(csv_path, encoding='windows-1252')

print(f"Raw dataset shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()

## 4. Data Quality Audit (Missing Values & Duplicates)

In [ ]:
print("=== Missing Values Audit ===")
null_counts = df_raw.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No missing values found across all columns.")

print("\n=== Duplicate Records Audit ===")
print(f"Exact duplicate rows: {df_raw.duplicated().sum()}")
print(f"Unique Orders: {df_raw['Order ID'].nunique():,}")
print(f"Unique Customers: {df_raw['Customer ID'].nunique():,}")
print(f"Unique Products: {df_raw['Product ID'].nunique():,}")

## 5. Data Cleaning & Feature Engineering

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()

# Safe date parsing
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')

# Numerical standardization
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce').fillna(0.0)
df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce').fillna(0.0)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').fillna(1).astype(int)
df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce').fillna(0.0)

# Feature Engineering
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.strftime('%B')
df['YearMonth'] = df['Order Date'].dt.strftime('%Y-%m')
df['Quarter'] = 'Q' + df['Order Date'].dt.quarter.astype(str)
df['Order Processing Days'] = (df['Ship Date'] - df['Order Date']).dt.days.fillna(0).astype(int)
df['Row Profit Margin'] = np.where(df['Sales'] > 0, df['Profit'] / df['Sales'], 0.0)

print("Engineered features added: Year, Month, Month Name, YearMonth, Quarter, Order Processing Days, Row Profit Margin.")
df[['Order ID', 'Order Date', 'Ship Date', 'Order Processing Days', 'Sales', 'Profit', 'Row Profit Margin']].head()

## 6. Business KPI Computation
Key Performance Indicators calculated with mathematical rigor:

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
# Note: Total profit margin = total profit / total sales * 100 (never average row margins!)
profit_margin = (total_profit / total_sales) * 100 if total_sales > 0 else 0.0
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()
aov = total_sales / total_orders if total_orders > 0 else 0.0

print(f"Total Sales:         ${total_sales:,.2f}")
print(f"Total Profit:        ${total_profit:,.2f}")
print(f"Overall Margin:      {profit_margin:.2f}%")
print(f"Total Orders:        {total_orders:,}")
print(f"Total Customers:     {total_customers:,}")
print(f"Average Order Value: ${aov:,.2f}")

## 7. Exploratory Data Analysis & Visualizations
### 7.1 Monthly Sales and Profit Trends

In [ ]:
monthly_perf = df.groupby('YearMonth').agg(sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly_perf['YearMonth'], monthly_perf['sales'], marker='o', color='#2563eb', label='Sales ($)', linewidth=2)
ax1.set_xlabel('Year-Month')
ax1.set_ylabel('Sales ($)', color='#2563eb')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly_perf['YearMonth'], monthly_perf['profit'], marker='s', color='#16a34a', label='Profit ($)', linewidth=2)
ax2.set_ylabel('Profit ($)', color='#16a34a')

plt.title('Monthly Sales and Net Profit Evolution (Chronological)')
fig.tight_layout()
plt.show()

### 7.2 Category & Sub-Category Performance

In [ ]:
cat_summary = df.groupby('Category').agg(sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index()
cat_summary['margin'] = cat_summary['profit'] / cat_summary['sales'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=cat_summary, x='Category', y='sales', ax=axes[0], palette='Blues_r')
axes[0].set_title('Total Sales by Category')
axes[0].set_ylabel('Sales ($)')

sns.barplot(data=cat_summary, x='Category', y='profit', ax=axes[1], palette='Greens_r')
axes[1].set_title('Net Profit by Category')
axes[1].set_ylabel('Profit ($)')
plt.tight_layout()
plt.show()

### 7.3 Regional and State Profitability

In [ ]:
reg_summary = df.groupby('Region').agg(sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index()
reg_summary['margin'] = reg_summary['profit'] / reg_summary['sales'] * 100
print(reg_summary)

## 8. Statistical Analysis & Hypothesis Testing
We examine the correlation between Sales, Discount, Profit, and Shipping Latency.

In [ ]:
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Order Processing Days']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Feature Correlation Matrix')
plt.show()

disc_profit_r, p_val = stats.pearsonr(df['Discount'], df['Profit'])
print(f"Pearson correlation between Discount and Profit: {disc_profit_r:.3f} (p-value: {p_val:.2e})")

## 9. Machine Learning: Chronological Time-Series Sales Forecasting
We train a Ridge Regression model with trend, autoregressive lags, and month-of-year seasonality.

In [ ]:
# Monthly aggregation
ts = df.groupby('YearMonth')['Sales'].sum().reset_index().sort_values('YearMonth')
ts['trend'] = np.arange(len(ts))
ts['lag_1'] = ts['Sales'].shift(1)
ts['lag_2'] = ts['Sales'].shift(2)
dates = pd.to_datetime(ts['YearMonth'] + '-01')
ts['month_sin'] = np.sin(2 * np.pi * dates.dt.month / 12)
ts['month_cos'] = np.cos(2 * np.pi * dates.dt.month / 12)

valid_ts = ts.dropna().reset_index(drop=True)
split_idx = int(len(valid_ts) * 0.8)

train = valid_ts.iloc[:split_idx]
test = valid_ts.iloc[split_idx:]

features = ['trend', 'lag_1', 'lag_2', 'month_sin', 'month_cos']
X_train, y_train = train[features], train['Sales']
X_test, y_test = test[features], test['Sales']

# Baseline: Naive lag-1
y_base_pred = test['lag_1']
base_rmse = root_mean_squared_error(y_test, y_base_pred)

# Ridge Regression
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
y_pred = np.maximum(0, model.predict(X_test))
model_rmse = root_mean_squared_error(y_test, y_pred)

print(f"Baseline Naive RMSE:   ${base_rmse:,.2f}")
print(f"Ridge Regression RMSE: ${model_rmse:,.2f}")
print(f"Error Reduction:       {((base_rmse - model_rmse) / base_rmse * 100):.1f}%")

## 10. Data-Driven Business Insights & Recommendations
1. **Eliminate Deep Discounts on Low-Margin Categories:** High discounts (>20%) sharply diminish net profit without generating compensatory volume.
2. **Address Sub-Category Losses:** Tables and Bookcases frequently post negative net margins due to shipping weight and steep discounting.
3. **Targeted Regional Strategy:** The West and East regions generate healthy profits, while specific Central and Southern states face chronic profit drag.
4. **Logistics Optimization:** Standard Class accommodates the vast majority of orders. Negotiating bulk contracts can improve gross margins.

## 11. Conclusion
This exploratory and predictive analysis validates that sales volume alone does not dictate profitability. By adhering to disciplined discounting, restructuring loss-making sub-categories, and utilizing time-series forecasting for inventory planning, the Superstore can reliably expand operating margins.